In [4]:
import torch
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

2.7.0+cu128
True
12.8
NVIDIA GeForce RTX 3080 Ti


In [2]:
# Load the model and tokenizer locally
model_name = "google/flan-t5-base"  # You can also use "google/flan-t5-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Create a text generation pipeline
pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,  # Uses lower precision for efficiency
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

e:\github\GenAI_HF_OpenAI\Hugging_Face_OpenAI_Azure\Code_Examples\.venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [3]:
#To control generation settings
generation_kwargs = {
    "temperature": 0.7,   # Controls randomness (higher = more creative)
    #"max_length": 512,    # Max number of tokens in response
    "max_new_tokens": 20, # max tokens
    #"min_length": 150,      # Forces at least 150 words (~150 tokens)
    "top_p": 0.9,         # Nucleus sampling (higher = more diverse responses)
    "top_k": 50,          # Limits the number of top tokens considered
    "repetition_penalty": 1.2,  # Penalizes repetition (1.0 = no penalty)
    "do_sample": True,    # Enables sampling (for creative responses)
}

In [5]:
# Wrap pipeline in LangChain's HuggingFacePipeline with parameters
llm = HuggingFacePipeline(pipeline=pipe, model_kwargs=generation_kwargs)

In [6]:
# Define prompt template
template = """Question: {question}
Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=["question"])

In [7]:
# Example questions
questions = [
    "Explain the concept of black holes in simple terms.",
    "What are the main causes of climate change, and how can we address them?",
    "Provide a brief overview of the history of artificial intelligence."
]

In [ ]:
#RunnableSequence
# from langchain.schema.runnable import RunnableSequence
chain = prompt | llm

for q in questions:
     print(f"\nQ: {q}")
     print(chain.invoke({"question": q}))